# 5.1 — Correlation Analysis: Latent Dimensions × Clinical Factors
**Thesis:** Unsupervised Deep Learning for Parkinson's Disease Imaging  
**Author:** Natnael Solomon Gebremichael — UniMiB MSc Data Science  

## What this notebook does
1. Loads the clean train/val files from notebook 5.0  
2. Filters to active latent dimensions (KL/variance threshold)  
3. Tests correlation between each active dimension and 12 clinical/demographic factors  
4. Applies Bonferroni correction for multiple comparisons  
5. Validates significant findings on the patient-stratified held-out set  
6. Runs ANCOVA to separate scanner effects from biological signal  
7. Produces the correlation heatmap and ANCOVA scatter plot  

## Improvements over baseline (Mahmoud's 5.1.1)
- 12 factors tested vs 8 — adds UPDRS I–IV, MoCA, RBD score, Hoehn & Yahr  
- SBR PCA loaded from saved objects — not refitted  
- Patient-stratified validation set — no data leakage  
- Full biological covariates in ANCOVA — not just SBR_PC1  


## 1. Import and Configuration

In [45]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
warnings.filterwarnings('ignore')


# File paths
TRAIN_FILE = '../../data/processed/clinical_merged/train.csv'
VAL_FILE = '../../data/processed/clinical_merged/val.csv'
SCALER_PATH = '../../results/models/scaler_sbr.pkl'
PCA_PATH = '../../results/models/pca_sbr.pkl'

OUTPUT_DIR = '../../results/correlation'
FIGURES_DIR = '../../results/figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Parameters
PATIENT_COL        = 'PATNO'
LABEL_COL          = 'label'
LATENT_COLS        = [f'latent_{i}' for i in range(256)]
VARIANCE_THRESHOLD = 0.24    # dims with std below this = collapsed
ALPHA              = 0.05   # base significance level
VAL_ALPHA          = 0.1    # lenient threshold for validation confirmation


# Clinical Factors
 # Continuous -> Pearson r^2
CONTINUOUS_FACTORS = [
    'AGE_AT_VISIT',
    'SBR_PC1', 'SBR_PC2', 'SBR_PC3',
    'UPDRS1_TOTAL', 'UPDRS2_TOTAL', 'UPDRS3_TOTAL', 'UPDRS4_TOTAL',
    'MOCA_TOTAL', 'RBD_SCORE',
]

 # Categorical
CATEGORICAL_FACTORS = ['SEX', 'HANDED', 'Manufacturer', 'HOEHN_YAHR']

ALL_FACTORS = CONTINUOUS_FACTORS + CATEGORICAL_FACTORS

print("Configuration loaded.")
print(f"  Continuous factors:  {len(CONTINUOUS_FACTORS)}")
print(f"  Categorical factors: {len(CATEGORICAL_FACTORS)}")
print(f"  Total factors:       {len(ALL_FACTORS)}")

Configuration loaded.
  Continuous factors:  10
  Categorical factors: 4
  Total factors:       14


## 2. Load Data

In [46]:
df_train = pd.read_csv(TRAIN_FILE)
df_val   = pd.read_csv(VAL_FILE)

# Load pre-fitted SBR PCA objects
with open(SCALER_PATH, 'rb') as f: scaler_sbr = pickle.load(f)
with open(PCA_PATH,    'rb') as f: pca_sbr    = pickle.load(f)

print("Data loaded")
print(f"Train: {df_train.shape}, {df_train[PATIENT_COL].nunique()} patients")
print(f"Val:   {df_val.shape},   {df_val[PATIENT_COL].nunique()} patients")

print(f"\nLabel distribution (train):")
print(df_train[LABEL_COL].value_counts().to_string())

print(f"\nFactor availability in train:")
for f in ALL_FACTORS:
    if f in df_train.columns:
        pct = df_train[f].notna().mean() * 100
        print(f"  {f:<25} {pct:.1f}%")
    else:
        print(f"  {f:<25} MISSING")

Data loaded
Train: (2221, 315), 1148 patients
Val:   (545, 315),   289 patients

Label distribution (train):
label
PD         1948
Control     185
SWEDD        88

Factor availability in train:
  AGE_AT_VISIT              99.9%
  SBR_PC1                   100.0%
  SBR_PC2                   100.0%
  SBR_PC3                   100.0%
  UPDRS1_TOTAL              67.4%
  UPDRS2_TOTAL              67.4%
  UPDRS3_TOTAL              67.3%
  UPDRS4_TOTAL              39.9%
  MOCA_TOTAL                73.8%
  RBD_SCORE                 48.5%
  SEX                       99.9%
  HANDED                    99.9%
  Manufacturer              100.0%
  HOEHN_YAHR                67.3%


## 3. Filter to Activate Latent Dimension
Collapsed dimensions (std < threshold) carry no information and would inflate 
the multiple comparison burden. They are excluded before any statistical testing

In [47]:
active_dims = [c for c in LATENT_COLS if df_train[c].std() > VARIANCE_THRESHOLD]
collapsed_dims = [c for c in LATENT_COLS if c not in active_dims]

print(f"Total dimensions:  {len(LATENT_COLS)}")
print(f"Active dimensions: {len(active_dims)}")
print(f"Collapsed:         {len(collapsed_dims)}")
print(f"Collapse rate:     {len(collapsed_dims)/len(LATENT_COLS)*100:.1f}%")

print(f"\nVariance distribution of active dims:")
stds = pd.Series([df_train[c].std() for c in active_dims])
print(f"  min={stds.min():.3f}  median={stds.median():.3f}  max={stds.max():.3f}")

Total dimensions:  256
Active dimensions: 66
Collapsed:         190
Collapse rate:     74.2%

Variance distribution of active dims:
  min=0.243  median=0.520  max=1.327


## 4. Bonferroni Correction

In [48]:
n_tests    = len(active_dims)
alpha_bonf = ALPHA / n_tests

print(f"Active dimensions (= number of tests per factor): {n_tests}")
print(f"Base alpha:       {ALPHA}")
print(f"Bonferroni alpha: {ALPHA}/{n_tests} = {alpha_bonf:.6f}")
# print(f"\nInterpretation: a dimension must achieve p < {alpha_bonf:.6f}")
# print(f"to be considered significant after correction.")

Active dimensions (= number of tests per factor): 66
Base alpha:       0.05
Bonferroni alpha: 0.05/66 = 0.000758


## 5. Correlation Tests — Training Set
For each active dimension × each factor:  
- **Continuous factors** → Pearson r (effect size = r^2)  
- **Categorical factors** → one-way ANOVA (effect size = η^2)


In [49]:
def pearson_test(series_a, series_b):
    valid = pd.concat([series_a, series_b], axis=1).dropna()
    if len(valid) < 10:
        return np.nan, np.nan, np.nan
    r, p = stats.pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
    return r, r**2, p

# One-way ANOVA
def eta_squared_test(df, dim_col, group_col):
    groups = [df[df[group_col] == g][dim_col].dropna().values
              for g in df[group_col].dropna().unique()]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) < 2:
        return np.nan, np.nan
    _, p       = stats.f_oneway(*groups)
    grand_mean = df[dim_col].mean()
    ss_total   = ((df[dim_col] - grand_mean)**2).sum()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    eta2       = ss_between / ss_total if ss_total > 0 else 0
    return eta2, p

print("Running correlation tests...")
train_results = []

for dim in active_dims:
    row = {'dimension': dim}

    for factor in CONTINUOUS_FACTORS:
        if factor not in df_train.columns:
            row[f'{factor}_r']   = np.nan
            row[f'{factor}_r2']  = np.nan
            row[f'{factor}_p']   = np.nan
            row[f'{factor}_sig'] = False
            continue
        r, r2, p = pearson_test(df_train[dim], df_train[factor])
        row[f'{factor}_r']   = round(r,  4) if not np.isnan(r) else np.nan
        row[f'{factor}_r2']  = round(r2, 4) if not np.isnan(r2) else np.nan
        row[f'{factor}_p']   = round(p,  6) if not np.isnan(p) else np.nan
        row[f'{factor}_sig'] = bool(p < alpha_bonf) if not np.isnan(p) else False

    for factor in CATEGORICAL_FACTORS:
        if factor not in df_train.columns:
            row[f'{factor}_eta2'] = np.nan
            row[f'{factor}_p']    = np.nan
            row[f'{factor}_sig']  = False
            continue
        eta2, p = eta_squared_test(df_train, dim, factor)
        row[f'{factor}_eta2'] = round(eta2, 4) if not np.isnan(eta2) else np.nan
        row[f'{factor}_p']    = round(p,    6) if not np.isnan(p)    else np.nan
        row[f'{factor}_sig']  = bool(p < alpha_bonf) if not np.isnan(p) else False

    train_results.append(row)

df_results = pd.DataFrame(train_results)

# Summary of significant findings
print("\n Significant dimensions per factor (Bonferroni corrected)")
for factor in CONTINUOUS_FACTORS:
    col = f'{factor}_sig'
    if col in df_results.columns:
        n_sig = df_results[col].sum()
        print(f"  {factor:<25} {n_sig:>3} / {len(active_dims)} dims significant")
for factor in CATEGORICAL_FACTORS:
    col = f'{factor}_sig'
    if col in df_results.columns:
        n_sig = df_results[col].sum()
        print(f"  {factor:<25} {n_sig:>3} / {len(active_dims)} dims significant")

df_results.to_csv(f'{OUTPUT_DIR}/correlation_train.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR}/correlation_train.csv")

Running correlation tests...

 Significant dimensions per factor (Bonferroni corrected)
  AGE_AT_VISIT               34 / 66 dims significant
  SBR_PC1                    30 / 66 dims significant
  SBR_PC2                    27 / 66 dims significant
  SBR_PC3                    42 / 66 dims significant
  UPDRS1_TOTAL               12 / 66 dims significant
  UPDRS2_TOTAL               10 / 66 dims significant
  UPDRS3_TOTAL               11 / 66 dims significant
  UPDRS4_TOTAL                8 / 66 dims significant
  MOCA_TOTAL                 18 / 66 dims significant
  RBD_SCORE                  12 / 66 dims significant
  SEX                        31 / 66 dims significant
  HANDED                      4 / 66 dims significant
  Manufacturer               65 / 66 dims significant
  HOEHN_YAHR                 15 / 66 dims significant

Saved: ../../results/correlation/correlation_train.csv


## 6. Validate on Held-Out Set
A result is **validated** if:  
1. It survived Bonferroni correction in training  
2. p < 0.1 in the validation set  
3. Direction (sign) of the correlation is the same in both splits


In [61]:
val_results = []

for _, train_row in df_results.iterrows():
    dim   = train_row['dimension']
    v_row = {'dimension': dim}
   
    for factor in CONTINUOUS_FACTORS:
        # print(f'{factor}_sig ',train_row.get(f'{factor}_sig'))
        # print(train_row)
        # print("-"*40)
        if not train_row.get(f'{factor}_sig', False):
            v_row[f'{factor}_validated'] = False
            continue
        if factor not in df_val.columns:
            v_row[f'{factor}_validated'] = False
            continue
        r_val, _, p_val = pearson_test(df_val[dim], df_val[factor])
        if np.isnan(r_val):
            v_row[f'{factor}_validated'] = False
            continue
        train_r   = train_row.get(f'{factor}_r', 0) or 0
        same_sign = (train_r * r_val) > 0
        v_row[f'{factor}_validated'] = bool(same_sign and p_val < VAL_ALPHA)    

    for factor in CATEGORICAL_FACTORS:
        if not train_row.get(f'{factor}_sig', False):
            v_row[f'{factor}_validated'] = False
            continue
        if factor not in df_val.columns:
            v_row[f'{factor}_validated'] = False
            continue
        groups_val = [df_val[df_val[factor] == g][dim].dropna().values
                      for g in df_val[factor].dropna().unique()
                      if len(df_val[df_val[factor] == g]) >= 3]
        if len(groups_val) < 2:
            v_row[f'{factor}_validated'] = False
            continue
        _, p_val = stats.f_oneway(*groups_val)
        v_row[f'{factor}_validated'] = bool(p_val < VAL_ALPHA)

    val_results.append(v_row)

df_validated = pd.DataFrame(val_results)

print("Validated dimensions per factor")
for factor in ALL_FACTORS:
    col = f'{factor}_validated'
    if col in df_validated.columns:
        n_val = df_validated[col].sum()
        n_sig = df_results[f'{factor}_sig'].sum()
        print(f"  {factor:<25} {n_val:>3} / {n_sig} significant dims validated")

df_validated.to_csv(f'{OUTPUT_DIR}/correlation_val.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR}/correlation_val.csv")

Validated dimensions per factor
  AGE_AT_VISIT               26 / 34 significant dims validated
  SBR_PC1                    20 / 30 significant dims validated
  SBR_PC2                    21 / 27 significant dims validated
  SBR_PC3                    36 / 42 significant dims validated
  UPDRS1_TOTAL                6 / 12 significant dims validated
  UPDRS2_TOTAL                7 / 10 significant dims validated
  UPDRS3_TOTAL                5 / 11 significant dims validated
  UPDRS4_TOTAL                0 / 8 significant dims validated
  MOCA_TOTAL                  7 / 18 significant dims validated
  RBD_SCORE                   4 / 12 significant dims validated
  SEX                        21 / 31 significant dims validated
  HANDED                      0 / 4 significant dims validated
  Manufacturer               61 / 65 significant dims validated
  HOEHN_YAHR                 10 / 15 significant dims validated

Saved: ../../results/correlation/correlation_val.csv
